# Class defining general configurations for the XAI explainers

In [1]:
# Defs for 
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        LRP
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion
#        LIME
#        KernelSHAP
#        GradientSHAP

In [ ]:
import sys

!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install captum

In [3]:
import numpy as np
import pandas as pd
import nbimporter
import time

# Utils
import torch
import torch.nn as nn
import os

# Captum XAI methods 
from captum.attr import IntegratedGradients
from captum.attr import InputXGradient
from captum.attr import DeepLift
from captum.attr import LRP
from captum.attr import NoiseTunnel
from captum.attr import Saliency
from captum.attr import GuidedBackprop
from captum.attr import Occlusion
from captum.attr import Lime, LimeBase
from captum.attr import KernelShap
from captum.attr import GradientShap

In [77]:
class Explainers:
    
    'Class defining general configurations for the XAI explainers we are using.'
        
    ############################################# methods from captum library
    
    #integrated gradients
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def int_grad(self, model, x):
        
        itGd= IntegratedGradients(model)
        x.requires_grad_()
        itGd_x_exp= (itGd.attribute(x)).squeeze().detach()
        
        return itGd_x_exp
    

    #input x gradient
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def inx_grad(self, model, x):
        
        iXGd= InputXGradient(model)
        x.requires_grad_()
        iXGd_x_exp= (iXGd.attribute(x.unsqueeze(0))).squeeze().detach()
        
        return iXGd_x_exp
    
    
    #deepLIFT
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def dp_lift(self, model, x):
        dLif= DeepLift(model)
        x.requires_grad_()
        dLif_x_exp= (dLif.attribute(x.unsqueeze(0))).squeeze().detach()
        
        return dLif_x_exp
    
    
    #LRP
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def lrp(self, model, x):
        lwrp= LRP(model)
        x.requires_grad_()
        lwrp_x_exp= (lwrp.attribute(x.unsqueeze(0))).squeeze().detach()
        
        return lwrp_x_exp

    
    #smoothgrad
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # if method=True it uses Vanilla Gradient, if method=False it uses Integrated Gradients
    # RETURN importances as a tensor([i1, ..., in])
    def smo_grad(self, model, x, method:bool=True):

        x.requires_grad_()
        
        if method: 
            nt= NoiseTunnel(Saliency(model))
            smoo_x_exp= (nt.attribute(x.unsqueeze(0), nt_type='smoothgrad', stdevs=0.05, nt_samples=10,
                                      abs=False)).squeeze().detach()
        else: 
            nt= NoiseTunnel(IntegratedGradients(model))
            smoo_x_exp= (nt.attribute(x.unsqueeze(0), nt_type='smoothgrad', stdevs=0.05, nt_samples=10, 
                                      baselines=torch.zeros(x.shape),
                                      return_convergence_delta=False)).squeeze().detach()

        return smoo_x_exp
    
    
    #vanilla gradient
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def vnl_grad(self, model, x):

        vnGd= Saliency(model)
        x.requires_grad_()
        vnGd_x_exp= (vnGd.attribute(x.unsqueeze(0), abs=False)).squeeze().detach()
        
        return vnGd_x_exp

    
    #guided backpropagation
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def g_bkprop(self, model, x):

        gdBp= GuidedBackprop(model)
        x.requires_grad_()
        gdBp_x_exp= (gdBp.attribute(x.unsqueeze(0))).squeeze().detach()
        
        return gdBp_x_exp

    
    #occlusion
    # model is a PyTorch Neural Network model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def occ(self, model, x):
        
        occl= Occlusion(model)
        occl_x_exp= (occl.attribute(x.unsqueeze(0), sliding_window_shapes=(1,1), 
                                    perturbations_per_eval=3)).squeeze().detach()
        
        return occl_x_exp

    
    #lime
    # model is a PyTorch model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def lime(self, model, x):
        
        lm= Lime(model)
        lime_x_exp= (lm.attribute(x.unsqueeze(0), n_samples=100)).squeeze().detach()
        
        return lime_x_exp

    
    #kernelSHAP
    # model is a PyTorch model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def k_shap(self, model, x):
        
        kshap= KernelShap(model)
        kshap_x_exp= (kshap.attribute(x.unsqueeze(0), n_samples=100)).squeeze().detach()
        
        return kshap_x_exp
    
    
    # gradientSHAP
    # model is a PyTorch model
    # x is a tensor([[x1, ..., xn]])
    # RETURN importances as a tensor([i1, ..., in])
    def g_shap(self, model, x):
        
        n_fts= x.size()[1]
        
        baseline_dist= torch.randn(10, n_fts) * 0.001
        
        x.requires_grad_()
        
        gshap= GradientShap(model)
        gshap_x_exp, delta= gshap.attribute(x, stdevs=0.05, n_samples=10,
                                            baselines=baseline_dist.double(),
                                            return_convergence_delta=True)
        
        return gshap_x_exp.squeeze().detach()

# Tests

In [4]:
import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

synth_ox= pd.read_csv('data/synth_OX_20.csv')

# split synth into features (x) and target (y)
df_inputs= synth_ox.loc[:,synth_ox.columns[0:20]]
df_labels= synth_ox.loc[:,synth_ox.columns[20:21]]

# split df_inputs and df_labels into train (80%) and test (20%) datasets
train_ox, test_ox, labels_train_ox, labels_test_ox= sklearn.model_selection.train_test_split(df_inputs,
                                                                                             df_labels,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)

In [5]:
import xgboost as xgb
from sklearn.neural_network import MLPClassifier

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ox= MLPClassifier(activation='relu', alpha=0.0001, hidden_layer_sizes=(64, 64, 64), 
                            learning_rate_init=0.01, max_iter=500, random_state=0, solver='sgd')

nn3_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn3_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn3_model_ox.predict(test_ox))
acc_nn3_ox

0.835

In [6]:
import pickle
from sklearn.base import clone

# Create a custom PyTorch model that mimics the behavior of a scikit-learn MLPClassifier model

import torch.nn as nn

# Define the PyTorch Neural Network model
class MLPClassifierModel(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(MLPClassifierModel, self).__init__()
        
        self.layers= nn.ModuleList([nn.Linear(input_size, hidden_sizes[0])])
        self.activations= [nn.ReLU()]
        
        for i in range(1, len(hidden_sizes)):
            self.layers.append(nn.Linear(hidden_sizes[i-1], hidden_sizes[i]))
            self.activations.append(nn.ReLU())
        
        self.output_layer= nn.Linear(hidden_sizes[-1], output_size)

        
    def forward(self, x):
        for layer, activation in zip(self.layers, self.activations):
            x= activation(layer(x))
        
        x= self.output_layer(x)
        
        return x

In [67]:
# convert a scikit-learn NN model to a PyTorch NN model

# skl_nn_model is the scikit-learn Neural Net model
# input_size is the number of input features

# RETURN a PyTorch Neural Net model used to binary classifications

def sklearn_to_pytorch_NN(skl_nn_model, input_size):

    # Convert the scikit-learn model to a PyTorch model
    input_size= input_size
    hidden_sizes= skl_nn_model.hidden_layer_sizes
    output_size= 1  # binary classification -- one output neuron for the binary prediction

    nn_pytorch_model= MLPClassifierModel(input_size, hidden_sizes, output_size)

    # Transfer the weights from the scikit-learn model to the PyTorch model
    for i, layer in enumerate(nn_pytorch_model.layers):
        layer.weight.data= torch.tensor(skl_nn_model.coefs_[i].T, dtype=torch.float64)
        layer.bias.data= torch.tensor(skl_nn_model.intercepts_[i], dtype=torch.float64)

    nn_pytorch_model.output_layer.weight.data= torch.tensor(skl_nn_model.coefs_[-1].T, dtype=torch.float64)
    nn_pytorch_model.output_layer.bias.data= torch.tensor(skl_nn_model.intercepts_[-1], dtype=torch.float64)
    
    
    return nn_pytorch_model

In [68]:
nn_pytorch_model_ox= sklearn_to_pytorch_NN(nn3_model_ox, train_ox.shape[1])

target_i= pd.DataFrame(data=[train_ox.iloc[0,:]], columns=train_ox.columns)
target_l= pd.DataFrame(data=[labels_train_ox.iloc[0]], columns=labels_train_ox.columns)

x_data_tensor= torch.tensor(np.asarray(target_i), dtype=torch.float64)

In [78]:
exp= Explainers()

In [16]:
exp.inx_grad(nn_pytorch_model_ox, x_data_tensor)

tensor([-6.6575,  1.8752, -6.0496,  4.5113,  0.1354,  1.4713, -0.0338, -0.4076,
         1.3699, -2.8737,  5.1172,  1.0209,  0.3882,  1.8144,  0.0570, -0.9116,
        -0.5277, -0.9357,  6.2458,  0.6547])

In [64]:
exp.dp_lift(nn_pytorch_model_ox, x_data_tensor)

tensor([-6.6575,  1.8752, -6.0496,  4.5113,  0.1354,  1.4713, -0.0338, -0.4076,
         1.3699, -2.8737,  5.1172,  1.0209,  0.3882,  1.8144,  0.0570, -0.9116,
        -0.5277, -0.9357,  6.2458,  0.6547])

In [18]:
exp.lrp(nn_pytorch_model_ox, x_data_tensor)

tensor([-6.6575,  1.8752, -6.0496,  4.5113,  0.1354,  1.4713, -0.0338, -0.4076,
         1.3699, -2.8737,  5.1172,  1.0209,  0.3882,  1.8144,  0.0570, -0.9116,
        -0.5277, -0.9357,  6.2458,  0.6547])

In [19]:
exp.smo_grad(nn_pytorch_model_ox, x_data_tensor)

tensor([ -6.7094,   5.8434, -12.5806,  12.5941,  -3.0148,   2.7544,  -0.0161,
         -1.9111,   5.7057, -10.2372,   5.3815,   1.3579,   1.2952,   3.3819,
          2.6565,  -0.5289,  -5.8985,  -0.9888,  12.5885,   0.8427])

In [20]:
exp.smo_grad(nn_pytorch_model_ox, x_data_tensor, False)

tensor([ 2.5239, -1.2368, -4.2779,  4.2447, -0.8600,  0.8717,  3.1276,  1.4353,
        -0.6710, -8.3534,  0.3520, -2.3688,  0.0745, -0.5269, -0.2100,  0.4308,
        -0.3414,  0.3775,  1.1003, -0.1349], dtype=torch.float64)

In [21]:
exp.vnl_grad(nn_pytorch_model_ox, x_data_tensor)

tensor([ -9.4376,   7.4746, -15.3771,  12.7608,   0.4124,   3.9255,  -0.1084,
         -3.1206,   4.8622,  -7.7922,   6.7460,   1.6791,   0.8806,   3.7649,
          0.2166,  -2.6158,  -4.1351,  -2.2249,  16.3407,   1.8816])

In [66]:
exp.g_bkprop(nn_pytorch_model_ox, x_data_tensor)

tensor([ -9.4376,   7.4746, -15.3771,  12.7608,   0.4124,   3.9255,  -0.1084,
         -3.1206,   4.8622,  -7.7922,   6.7460,   1.6791,   0.8806,   3.7649,
          0.2166,  -2.6158,  -4.1351,  -2.2249,  16.3407,   1.8816])

In [23]:
exp.occ(nn_pytorch_model_ox, x_data_tensor)

tensor([-3.2708,  1.0653, -4.8717,  2.0690, -0.9839,  1.3859, -0.2458, -0.6021,
         1.1064, -6.1630,  0.7070, -0.9935, -0.1417,  1.1071,  0.4104, -1.0770,
        -0.7904, -0.9089,  2.9651,  0.2560])

In [24]:
exp.lime(nn_pytorch_model_ox, x_data_tensor)

tensor([ 1.0276, -1.1989, -3.6688,  2.4081,  0.9115,  0.3536,  1.7390,  1.1134,
        -0.3692, -4.7223,  0.7945, -1.2436, -0.3570, -0.4525, -0.1203,  0.1194,
         0.1723,  0.5890,  1.3787,  0.4468])

In [25]:
exp.k_shap(nn_pytorch_model_ox, x_data_tensor)

tensor([ 1.7265, -1.3494, -3.6846,  3.6382, -0.4107,  0.3945,  2.4350,  0.9826,
        -1.3460, -5.9452,  0.5828, -1.7211,  0.3113, -0.7676, -0.6906, -0.5422,
        -0.3492, -0.2760,  2.5987, -0.4288])

In [79]:
exp.g_shap(nn_pytorch_model_ox, x_data_tensor)

tensor([-9.1584e-01, -2.1754e-01, -5.2107e+00,  4.2202e+00, -7.1209e-01,
         1.3498e+00,  2.2749e+00,  5.7725e-01, -2.4177e-02, -6.4081e+00,
         1.8236e+00, -1.2679e+00,  1.1170e-01,  4.9106e-01, -1.2999e-01,
         4.2040e-01, -5.0036e-01,  1.5372e-01,  2.6692e+00,  3.4769e-03],
       dtype=torch.float64)